# GNAS: Pseudohypoparathyroidism Ia vs Pseudopseudohypoparathyroidism

Cohort: `GNAS`, Phenopacket Store v0.1.27 (n = 103).

Disease composition:
- Pseudohypoparathyroidism Ia (PHP-Ia), [OMIM:103580](https://omim.org/entry/103580), n = 80
- Pseudopseudohypoparathyroidism (PPHP), [OMIM:612463](https://omim.org/entry/612463), n = 23

Both subtypes share the Albright hereditary osteodystrophy (AHO) features. They are
distinguished by parent-of-origin: maternally inherited inactivating GNAS variants cause
PHP-Ia with multi-hormone resistance, paternally inherited variants cause PPHP with AHO
alone. This makes disease membership a clinically meaningful synergy target.

Source publication: Thiele et al. 2015, Mol Genet Genomic Med 3:111-120 (PMID 25802881).


In [ ]:
from ppktstore.registry import configure_phenopacket_registry
from phenosign import (
    PhenotypeDatasetBuilder,
    HPOCorrelationAnalyzer,
    SynergyAnalyzer,
    has_disease,
)
import pandas as pd

In [ ]:
registry = configure_phenopacket_registry()
with registry.open_phenopacket_store(release="0.1.27") as ps:
    phenopackets = list(ps.iter_cohort_phenopackets("GNAS"))

print(f"Loaded {len(phenopackets)} phenopackets")

In [ ]:
# has_disease is evaluated on the phenopacket directly, so the (slow) GPSEA
# cohort build is not required for this analysis.
dataset = PhenotypeDatasetBuilder(phenopackets).build(build_gpsea_cohort=False)

In [ ]:
diseases_df, sex_df, genes_df, _ = dataset.describe_conditions()
diseases_df

## Pairwise correlation

In [ ]:
analyzer = HPOCorrelationAnalyzer(dataset=dataset)
results = analyzer.compute_correlation_matrix()

results.results_table.head()

In [ ]:
results.save_correlation_results(
    output_file="../case_studies/GNAS/correlation_results_GNAS.csv"
)

In [ ]:
sig = results.results_table.query("adj_p_value < 0.05")
sig = sig.reindex(sig["correlation"].abs().sort_values(ascending=False).index)
sig[["HPO_A_label", "HPO_B_label", "correlation", "adj_p_value", "n_individuals", "n_pmids"]]

In [ ]:
results.plot_correlation_heatmap_with_significance(
    title_name="Cohort GNAS",
)

## Synergy with respect to disease subtype

Target: PHP-Ia (OMIM:103580) as the positive class, PPHP as the negative class.

`min_individuals_for_synergy_calculation` is lowered from the default of 30 because the
cohort splits 80/23 and the smaller arm would otherwise exclude most candidate pairs.

In [ ]:
condition = dataset.get_condition(
    has_disease("OMIM:103580"),
    name="PHP-Ia vs PPHP",
)

condition.value_counts(dropna=False)

In [ ]:
synergy_analyzer = SynergyAnalyzer(
    dataset=dataset,
    min_individuals_for_synergy_calculation=25,
)
synergy_results = synergy_analyzer.compute_synergy_matrix(condition=condition)

synergy_results.results_table.head()

In [ ]:
# Note: save_synergy_results() filters at synergy >= 0.01 and therefore drops the
# negative (redundant) half of the distribution. Save the full table as well.
synergy_results.results_table.to_csv(
    "../case_studies/GNAS/synergy_full_GNAS_OMIM_103580.csv", index=False
)
synergy_results.save_synergy_results(
    output_file="../case_studies/GNAS/synergy_matrix_GNAS_OMIM_103580.csv"
)

In [ ]:
t = synergy_results.results_table
print("Top synergy")
display(t.nlargest(10, "synergy")[["HPO_A_label", "HPO_B_label", "synergy", "adj_p_value", "n_individuals"]])

print("Top dissynergy (redundancy)")
display(t.nsmallest(10, "synergy")[["HPO_A_label", "HPO_B_label", "synergy", "adj_p_value", "n_individuals"]])

In [ ]:
synergy_results.plot_synergy_heatmap()